In [ ]:
from resram_core import load_input, resram_data, param_init, raman_residual, run_save
from tqdm import tqdm
import time
from datetime import datetime
import numpy as np
import lmfit

import resram_core
# resram_core.HAS_RUST = False  # Disable Rust implementation for benchmarking

In [ ]:
# Count time for loading data and simulation
start_time = time.time()
for i in range(1):
    data = resram_data() #load input data from current directory and calculate cross sections
data.plot()
end_time = time.time()
elapsed_time = end_time - start_time
print(f"Elapsed time: {elapsed_time} seconds")

In [ ]:
start_time = time.time()
for i in range(100):
    fit_obj = load_input()
end_time = time.time()
elapsed_time = end_time - start_time
print(f"Elapsed time: {elapsed_time} seconds")
print(fit_obj.rp)

In [ ]:
# Initialize the 'fit_obj' object of class 'load_input'.
# Load from input files in root dir or specified dir.
fit_obj = load_input() 

fit_switch = np.ones(len(fit_obj.delta)+7)
fit_switch[len(fit_obj.delta)] = 1  # fit gamma?
fit_switch[len(fit_obj.delta)+1] = 1  # fit M?
fit_switch[len(fit_obj.delta)+2] = 1  # fit theta?
fit_switch[len(fit_obj.delta)+3] = 0  # fit kappa? We assume the slow-modulation limit, realistic for a polar solute in a polar solvent, with𝜅 = 0.1 so that the resulting lineshapes will be nearly Gaussian' in Zak's dissertation
fit_switch[len(fit_obj.delta)+5] = 0  # fit E0? 

params_lmfit = param_init(fit_switch, fit_obj)   # Initialize parameters for fitting

for name in params_lmfit.keys():
    print(params_lmfit[name])

# Initial call to populate fit_obj attributes
raman_residual(params_lmfit, fit_obj)
print(f"Initial correlation: {fit_obj.correlation}")

fit_kws = dict(tol=1e-10)    # Set fitting tolerance
max_nfev = 10000000    # Set maximum number of function evaluations

# Initialize tqdm progress bar for fitting process
with tqdm(total=max_nfev, desc="Fitting progress") as pbar:
    def update_progress(params, iteration, resid, *args, **kwargs):
        pbar.update(1)
        try:
            pbar.set_description(
                f"Iteration {iteration}, Loss: {resid[0]:.6f}, Abs_corr: {fit_obj.correlation:.6f}")
        except Exception:
            pass

    # Perform the fitting using lmfit.minimize function
    result = lmfit.minimize(raman_residual, params_lmfit, args=(fit_obj,), method='cobyla', **fit_kws,
                            max_nfev=max_nfev, iter_cb=update_progress)

print(lmfit.fit_report(result))

current_time_str = datetime.now().strftime("%Y%m%d_%H-%M-%S") 
output = run_save(fit_obj, current_time_str)  # save all parameters and results to a new folder

# Save history lists
np.savetxt(current_time_str+"_data/corr_list.dat", fit_obj.correlation_list)
np.savetxt(current_time_str+"_data/sigma_list.dat", fit_obj.sigma_list)
np.savetxt(current_time_str+"_data/loss_list.dat", fit_obj.loss_list)

print(f"Fitting completed. Results saved in {current_time_str}_data/")


In [ ]:
output.plot()

In [ ]:
output = resram_data("20250604_13-08-34_data")
print(output.abs)
output.plot()

